In [13]:
import pandas as pd
import zipfile
import os
import glob

# ── CONFIG ──────────────────────────────────────────────────────────────────
CENSUS_DIR = "/Users/aliabonanno/public_policy_project/census_data"   # <-- change this to your actual path
OUTPUT_PATH = "/Users/aliabonanno/public_policy_project/acs_poverty_panel.csv"


# Columns we need from S1701
# S1701_C01_001E = total population for whom poverty status is determined
# S1701_C01_040E = count with income below 150% of poverty line
KEEP_COLS = ["GEO_ID", "S1701_C01_001E", "S1701_C01_040E"]

# ── LOOP OVER ZIP FILES ──────────────────────────────────────────────────────
all_years = []

for zip_path in sorted(glob.glob(os.path.join(CENSUS_DIR, "ACSST5Y*.zip"))):

    # Extract year from filename, e.g. ACSST5Y2015.S1701_...zip -> 2015
    basename = os.path.basename(zip_path)
    year = int(basename[7:11])

    # Open zip and find the data CSV (not metadata, not notes)
    with zipfile.ZipFile(zip_path, "r") as z:
        data_file = [f for f in z.namelist() if "Data" in f and f.endswith(".csv")]
        if not data_file:
            print(f"WARNING: No data CSV found in {basename}, skipping.")
            continue
        
        with z.open(data_file[0]) as f:
            # Row 0 = column codes, Row 1 = human-readable labels, data starts row 2
            # We skip row 1 (the label row) by reading then dropping it
            df = pd.read_csv(f, skiprows=[1], dtype=str)

    # Check required columns exist
    missing = [c for c in KEEP_COLS if c not in df.columns]
    if missing:
        print(f"WARNING: {year} missing columns {missing}, skipping.")
        continue

    df = df[KEEP_COLS].copy()

    # Extract 5-digit FIPS from GEO_ID (format: 0500000US01001)
    # Keep only county-level rows (GEO_ID starts with 0500000US)
    df = df[df["GEO_ID"].str.startswith("0500000US")]
    df["FIPS"] = df["GEO_ID"].str[-5:]

    # Convert to numeric, coerce errors (handles "-", "N", "(X)" etc.)
    df["total_pop"] = pd.to_numeric(df["S1701_C01_001E"], errors="coerce")
    df["pop_below_150pct"] = pd.to_numeric(df["S1701_C01_040E"], errors="coerce")

    # Construct poverty rate: share of population below 150% poverty line
    df["pct_below_150pct_poverty"] = df["pop_below_150pct"] / df["total_pop"]

    # Tag with year (this is the end year of the 5-year window)
    df["year"] = year

    # Keep only clean columns
    df = df[["FIPS", "year", "total_pop", "pop_below_150pct", "pct_below_150pct_poverty"]]

    # Drop rows where FIPS is missing or poverty rate couldn't be computed
    df = df.dropna(subset=["FIPS", "pct_below_150pct_poverty"])

    all_years.append(df)
    print(f"  {year}: {len(df)} counties loaded.")

# ── STACK AND SAVE ───────────────────────────────────────────────────────────
panel = pd.concat(all_years, ignore_index=True)
panel = panel.sort_values(["FIPS", "year"]).reset_index(drop=True)

# Sanity checks
print(f"\nFinal panel: {len(panel)} rows")
print(f"Years covered: {sorted(panel['year'].unique())}")
print(f"Unique counties: {panel['FIPS'].nunique()}")
print(f"Missing poverty rate: {panel['pct_below_150pct_poverty'].isna().sum()}")
print(f"\nSample rows:")
print(panel.head(10).to_string())
print(f"\nSummary stats:")
print(panel["pct_below_150pct_poverty"].describe())

panel.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")


2012:
  S1701_C01_036E: Total!!Estimate!!All Individuals below!!150 percent of poverty level


ValueError: Attempt to use ZIP archive that was already closed

In [15]:
import pandas as pd
import zipfile
import os
import glob

CENSUS_DIR = "/Users/aliabonanno/public_policy_project/census_data"

for zip_path in sorted(glob.glob(os.path.join(CENSUS_DIR, "ACSST5Y*.zip"))):
    basename = os.path.basename(zip_path)
    year = int(basename[7:11])

    with zipfile.ZipFile(zip_path, "r") as z:
        data_file = [f for f in z.namelist() if "Data" in f and f.endswith(".csv")]
        if not data_file:
            continue
        with z.open(data_file[0]) as f:
            labels = pd.read_csv(f, nrows=1)

    matches = [(col, labels[col].values[0]) for col in labels.columns
               if "150" in str(labels[col].values[0])]

    print(f"\n{year}:")
    for col, label in matches:
        print(f"  {col}: {label}")


2012:
  S1701_C01_036E: Total!!Estimate!!All Individuals below!!150 percent of poverty level
  S1701_C01_036M: Total!!Margin of Error!!All Individuals below!!150 percent of poverty level
  S1701_C02_036E: Below poverty level!!Estimate!!All Individuals below!!150 percent of poverty level
  S1701_C02_036M: Below poverty level!!Margin of Error!!All Individuals below!!150 percent of poverty level
  S1701_C03_036E: Percent below poverty level!!Estimate!!All Individuals below!!150 percent of poverty level
  S1701_C03_036M: Percent below poverty level!!Margin of Error!!All Individuals below!!150 percent of poverty level

2013:
  S1701_C01_036E: Total!!Estimate!!All Individuals below!!150 percent of poverty level
  S1701_C01_036M: Total!!Margin of Error!!All Individuals below!!150 percent of poverty level
  S1701_C02_036E: Below poverty level!!Estimate!!All Individuals below!!150 percent of poverty level
  S1701_C02_036M: Below poverty level!!Margin of Error!!All Individuals below!!150 percen

In [17]:
import pandas as pd
import zipfile
import os
import glob

# ── CONFIG ───────────────────────────────────────────────────────────────────
CENSUS_DIR = "/Users/aliabonanno/public_policy_project/census_data"
OUTPUT_PATH = "/Users/aliabonanno/public_policy_project/acs_poverty_panel.csv"

# Column for total population (consistent across all years)
TOTAL_POP_COL = "S1701_C01_001E"

# Column for count below 150% poverty line — changed in 2015
def get_poverty_col(year):
    if year <= 2014:
        return "S1701_C01_036E"
    else:
        return "S1701_C01_040E"

# ── LOOP OVER ZIP FILES ───────────────────────────────────────────────────────
all_years = []

for zip_path in sorted(glob.glob(os.path.join(CENSUS_DIR, "ACSST5Y*.zip"))):

    basename = os.path.basename(zip_path)
    year = int(basename[7:11])
    poverty_col = get_poverty_col(year)
    keep_cols = ["GEO_ID", TOTAL_POP_COL, poverty_col]

    with zipfile.ZipFile(zip_path, "r") as z:
        data_file = [f for f in z.namelist() if "Data" in f and f.endswith(".csv")]
        if not data_file:
            print(f"WARNING: No data CSV found in {basename}, skipping.")
            continue
        with z.open(data_file[0]) as f:
            df = pd.read_csv(f, skiprows=[1], dtype=str)

    # Check required columns exist
    missing = [c for c in keep_cols if c not in df.columns]
    if missing:
        print(f"WARNING: {year} missing columns {missing}, skipping.")
        continue

    df = df[keep_cols].copy()

    # Keep only county-level rows and extract 5-digit FIPS
    df = df[df["GEO_ID"].str.startswith("0500000US")]
    df["FIPS"] = df["GEO_ID"].str[-5:]

    # Convert to numeric
    df["total_pop"] = pd.to_numeric(df[TOTAL_POP_COL], errors="coerce")
    df["pop_below_150pct"] = pd.to_numeric(df[poverty_col], errors="coerce")

    # Poverty rate
    df["pct_below_150pct_poverty"] = df["pop_below_150pct"] / df["total_pop"]

    df["year"] = year
    df = df[["FIPS", "year", "total_pop", "pop_below_150pct", "pct_below_150pct_poverty"]]
    df = df.dropna(subset=["FIPS", "pct_below_150pct_poverty"])

    all_years.append(df)
    print(f"  {year} (col: {poverty_col}): {len(df)} counties loaded.")

# ── STACK AND SAVE ────────────────────────────────────────────────────────────
panel = pd.concat(all_years, ignore_index=True)
panel = panel.sort_values(["FIPS", "year"]).reset_index(drop=True)

print(f"\nFinal panel: {len(panel)} rows")
print(f"Years covered: {sorted(panel['year'].unique())}")
print(f"Unique counties: {panel['FIPS'].nunique()}")
print(f"Missing poverty rate: {panel['pct_below_150pct_poverty'].isna().sum()}")

# Sanity check: verify the 2014-2015 jump is gone for Autauga County (01001)
autauga = panel[panel["FIPS"] == "01001"][["FIPS", "year", "pct_below_150pct_poverty"]]
print(f"\nAutauga County (01001) — should be smooth across 2014-2015:")
print(autauga.to_string(index=False))

print(f"\nSummary stats:")
print(panel["pct_below_150pct_poverty"].describe())

panel.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

  2012 (col: S1701_C01_036E): 3221 counties loaded.
  2013 (col: S1701_C01_036E): 3221 counties loaded.
  2014 (col: S1701_C01_036E): 3220 counties loaded.
  2015 (col: S1701_C01_040E): 3220 counties loaded.
  2016 (col: S1701_C01_040E): 3220 counties loaded.
  2017 (col: S1701_C01_040E): 3220 counties loaded.
  2018 (col: S1701_C01_040E): 3219 counties loaded.
  2019 (col: S1701_C01_040E): 3220 counties loaded.
  2020 (col: S1701_C01_040E): 3221 counties loaded.
  2021 (col: S1701_C01_040E): 3221 counties loaded.
  2022 (col: S1701_C01_040E): 3222 counties loaded.
  2023 (col: S1701_C01_040E): 3222 counties loaded.
  2024 (col: S1701_C01_040E): 3222 counties loaded.

Final panel: 41869 rows
Years covered: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Unique counties: 3234
Missing poverty rate: 0

Autauga County (01001) — should be smooth across 2014-2015:
 FIPS  year  pct_below_150pct_poverty
01001  2012                  0.190876
01001  2013           

In [13]:
import requests
import pandas as pd
import os
import numpy as np
import time

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project/census_data"
OUTPUT_PATH = "/Users/aliabonanno/public_policy_project/acs_controls_panel.csv"

# Census API base URL (no key needed for small requests, but add one if you get errors)
# Get a free key at: api.census.gov/data/key_signup.html
API_KEY = ""  # leave empty or add your key here

YEARS = list(range(2015, 2025))

# ── VARIABLES TO PULL ─────────────────────────────────────────────────────────
# B19013_001E = median household income
# B02001_001E = total population (race denominator)
# B02001_002E = white alone
# B02001_003E = Black alone
# B03002_012E = Hispanic or Latino (from B03002 table)
# S1501_C02_015E = % bachelor's degree or higher (25+)
# S2301_C04_001E = unemployment rate (16+)

# We pull from two datasets: acs/acs5 (detailed) and acs/acs5/subject (S tables)

DETAIL_VARS = {
    "B19013_001E": "median_hh_income",
    "B02001_001E": "pop_race_total",
    "B02001_002E": "pop_white",
    "B02001_003E": "pop_black",
    "B03002_012E": "pop_hispanic",
    "B25040_001E": "heating_total",
    "B25040_007E": "heating_electric",
}

# Subject table variables (pulled separately)
SUBJECT_VARS = {
    "S1501_C02_015E": "pct_bachelors_plus",
    "S2301_C04_001E": "unemployment_rate",
}

# ── HELPER ────────────────────────────────────────────────────────────────────
def fetch_acs(year, dataset, variables, geo="county:*"):
    var_str = ",".join(["NAME"] + list(variables.keys()))
    url = f"https://api.census.gov/data/{year}/acs/{dataset}"
    params = {"get": var_str, "for": geo}
    if API_KEY:
        params["key"] = API_KEY

    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        print(f"  ERROR {year} {dataset}: {r.status_code}")
        return None

    data = r.json()
    cols = data[0]
    rows = data[1:]
    df = pd.DataFrame(rows, columns=cols)
    return df

# ── LOOP OVER YEARS ───────────────────────────────────────────────────────────
all_years = []

for year in YEARS:
    print(f"\n{year}:")

    # Pull detail table variables
    df_detail = fetch_acs(year, "acs5", DETAIL_VARS)
    if df_detail is None:
        continue

    # Pull subject table variables
    df_subject = fetch_acs(year, "acs5/subject", SUBJECT_VARS)
    if df_subject is None:
        continue

    # Construct FIPS from state + county columns
    df_detail["FIPS"] = df_detail["state"].str.zfill(2) + df_detail["county"].str.zfill(3)
    df_subject["FIPS"] = df_subject["state"].str.zfill(2) + df_subject["county"].str.zfill(3)

    # Rename detail columns
    df_detail = df_detail.rename(columns=DETAIL_VARS)
    df_subject = df_subject.rename(columns=SUBJECT_VARS)

    # Keep only what we need
    detail_keep = ["FIPS"] + list(DETAIL_VARS.values())
    subject_keep = ["FIPS"] + list(SUBJECT_VARS.values())

    df = df_detail[detail_keep].merge(
        df_subject[subject_keep], on="FIPS", how="outer"
    )
    df["year"] = year

    # Convert to numeric
    for col in list(DETAIL_VARS.values()) + list(SUBJECT_VARS.values()):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Replace Census sentinel values with NaN
    df["median_hh_income"] = df["median_hh_income"].where(
    df["median_hh_income"] > 0, np.nan)

    # Construct share variables
    df["pct_white"]    = df["pop_white"]    / df["pop_race_total"]
    df["pct_black"]    = df["pop_black"]    / df["pop_race_total"]
    df["pct_hispanic"] = df["pop_hispanic"] / df["pop_race_total"]
    df["pct_electric_heat"] = df["heating_electric"] / df["heating_total"]

    # Keep final columns
    final_cols = ["FIPS", "year", "median_hh_income",
              "pct_white", "pct_black", "pct_hispanic",
              "pct_bachelors_plus", "unemployment_rate",
              "pct_electric_heat"]
    df = df[final_cols].dropna(subset=["FIPS"])

    all_years.append(df)
    print(f"  {len(df)} counties loaded")
    time.sleep(0.5)  # be polite to Census API

# ── STACK AND SAVE ────────────────────────────────────────────────────────────
panel = pd.concat(all_years, ignore_index=True)
panel = panel.sort_values(["FIPS", "year"]).reset_index(drop=True)

print(f"\nFinal controls panel: {len(panel)} rows")
print(f"Years: {sorted(panel['year'].unique())}")
print(f"Unique counties: {panel['FIPS'].nunique()}")
print(f"\nSample:")
print(panel[panel["FIPS"] == "01001"].to_string())
print(f"\nSummary stats:")
print(panel[["median_hh_income", "pct_white", "pct_black",
             "pct_hispanic", "pct_bachelors_plus",
             "unemployment_rate"]].describe().round(3))
print(panel[["median_hh_income", "pct_white", "pct_black",
             "pct_hispanic", "pct_bachelors_plus",
             "unemployment_rate", "pct_electric_heat"]].describe().round(3))

panel.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")


2015:
  3220 counties loaded

2016:
  3220 counties loaded

2017:
  3220 counties loaded

2018:
  3220 counties loaded

2019:
  3220 counties loaded

2020:
  3221 counties loaded

2021:
  3221 counties loaded

2022:
  3222 counties loaded

2023:
  3222 counties loaded

2024:
  3222 counties loaded

Final controls panel: 32208 rows
Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Unique counties: 3231

Sample:
    FIPS  year  median_hh_income  pct_white  pct_black  pct_hispanic  pct_bachelors_plus  unemployment_rate  pct_electric_heat
0  01001  2015           51281.0   0.773999   0.186795      0.026077                23.2                7.6           0.007992
1  01001  2016           53099.0   0.768606   0.185217      0.025723                24.6                5.6           0.009135
2  01001  2017           55317.0   0.766026   0.190966      0.026655                25.0                5.2           0.009072
3  01001  2018           58786.0   0.768786   0.191395     

In [19]:
import pandas as pd

sales = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861_sales.csv")
print("Raw ownership values in annual EIA 861:")
print(sales["ownership"].value_counts())
print(f"\nTotal utilities: {sales['utility_id'].nunique()}")

# Also check the monthly data
sales_m = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861m_sales.csv")
print("\nRaw ownership values in monthly EIA 861M:")
print(sales_m["ownership"].value_counts())

Raw ownership values in annual EIA 861:
ownership
Cooperative                    8325
Municipal                      8165
Retail Power Marketer          6703
Investor Owned                 2716
Behind the Meter               1901
Political Subdivision           712
Community Choice Aggregator     179
State                           105
Federal                          41
Name: count, dtype: int64

Total utilities: 3253

Raw ownership values in monthly EIA 861M:
ownership
Investor Owned           22419
Cooperative              14319
Municipal                 8406
Political Subdivision     2352
State                      839
Federal                    120
Name: count, dtype: int64


In [21]:
import pandas as pd

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

# One row per county
counties = master.drop_duplicates(subset=["FIPS"])

print("County counts by ownership type:")
print(counties["dominant_ownership_type"].value_counts())
print(f"\nTotal counties: {len(counties)}")
print(f"\nPercentages:")
print(counties["dominant_ownership_type"].value_counts(normalize=True).round(3))

County counts by ownership type:
dominant_ownership_type
IOU            2532
Cooperative     165
Municipal       117
Name: count, dtype: int64

Total counties: 2814

Percentages:
dominant_ownership_type
IOU            0.900
Cooperative    0.059
Municipal      0.042
Name: proportion, dtype: float64
